In [0]:
-- ############################################################################
-- Filename: setup/security_policies.sql
-- Purpose: Applies RLS to the Gold Dimension Table (Star Schema)
-- ############################################################################

USE CATALOG workspace;
USE SCHEMA dev_gold_layer;

-- 1. Create the filter function for brands
CREATE OR REPLACE FUNCTION brand_filter(brand_name STRING)
RETURN brand_name IS NOT NULL;

-- 2. Apply the filter to the Product Dimension table
-- This ensures that any join to 'fact_sales' will also be filtered.
ALTER TABLE dim_products 
SET ROW FILTER brand_filter ON (brand);

-- 1. Row Filter: Protects sensitive categories
CREATE OR REPLACE FUNCTION category_filter(category STRING)
RETURN is_account_group_member('regional_managers') OR category IS NOT NULL;

ALTER TABLE dim_products 
SET ROW FILTER category_filter ON (category_code);

-- 2. Column Mask: Protects PII (User IDs) in Fact Tables
-- Note: Fact tables often contain user_id which should be redacted for non-admins.
CREATE OR REPLACE FUNCTION user_id_mask(user_id STRING)
RETURN CASE 
    WHEN is_account_group_member('admin') THEN user_id 
    ELSE '####-REDACTED-####' 
END;

-- 3. Verification
DESCRIBE TABLE EXTENDED dim_products;